In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import seaborn as sns
from tqdm import tqdm
import json
import random
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import warnings
warnings.filterwarnings('ignore')

# Set working directory
os.makedirs("/content/nighttime_vehicle_detection", exist_ok=True)
os.chdir("/content/nighttime_vehicle_detection")
print("Working Directory:", os.getcwd())

drive_path = '/content/drive/MyDrive'
sub_dir = '/content/drive/MyDrive/nighttime_vehicle_results'

# Create results directory
os.makedirs(sub_dir, exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Working Directory: /content/nighttime_vehicle_detection
Using device: cuda


In [ ]:
class ImageEnhancer:
    """Traditional image processing + neural enhancement hybrid"""

    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

    def enhance_frame(self, frame):
        """Apply CLAHE + Gamma Correction"""
        # Convert to LAB color space
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)

        # Apply CLAHE to L channel
        l = self.clahe.apply(l)

        # Merge channels
        enhanced = cv2.merge([l, a, b])
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

        # Gamma correction
        gamma = 1.2
        enhanced = np.power(enhanced/255.0, gamma)
        enhanced = (enhanced * 255).astype(np.uint8)

        return enhanced


In [ ]:
class YOLOv8Backbone(nn.Module):
    """YOLOv8-style CNN backbone for spatial feature extraction"""

    def __init__(self, in_channels=3):
        super(YOLOv8Backbone, self).__init__()

        # C2f blocks (similar to YOLOv8)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 6, 2, 2),
            nn.BatchNorm2d(64),
            nn.SiLU(inplace=True)
        )

        self.stage1 = self._make_stage(64, 128, 2)
        self.stage2 = self._make_stage(128, 256, 2)
        self.stage3 = self._make_stage(256, 512, 2)
        self.stage4 = self._make_stage(512, 1024, 2)

        # Feature pyramid levels
        self.fpn_channels = [256, 512, 1024]

    def _make_stage(self, in_channels, out_channels, stride):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True)
        )

    def forward(self, x):
        """Extract multi-scale features"""
        x = self.stem(x)

        x1 = self.stage1(x)     # 1/4 scale
        x2 = self.stage2(x1)    # 1/8 scale
        x3 = self.stage3(x2)    # 1/16 scale
        x4 = self.stage4(x3)    # 1/32 scale

        return [x2, x3, x4]  # Return P3, P4, P5 features


In [ ]:
class ConvLSTMCell(nn.Module):
    """ConvLSTM cell for temporal modeling"""

    def __init__(self, input_dim, hidden_dim, kernel_size, bias=True):
        super(ConvLSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias

        self.conv = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=4 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=self.bias
        )

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state

        # Concatenate input and hidden state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)

        # Split into gates
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)

        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

class ConvLSTM(nn.Module):
    """ConvLSTM for temporal sequence modeling"""

    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers=1):
        super(ConvLSTM, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.num_layers = num_layers

        cell_list = []
        for i in range(num_layers):
            cur_input_dim = input_dim if i == 0 else hidden_dim
            cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dim, kernel_size))

        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, input_tensor, hidden_state=None):
        """
        input_tensor: (batch, seq_len, channels, height, width)
        """
        b, seq_len, c, h, w = input_tensor.size()

        if hidden_state is None:
            hidden_state = self._init_hidden(b, h, w)

        layer_output_list = []
        layer_hidden_list = []

        cur_layer_input = input_tensor

        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]
            output_inner = []

            for t in range(seq_len):
                h, c = self.cell_list[layer_idx](cur_layer_input[:, t, :, :, :], (h, c))
                output_inner.append(h)

            layer_output = torch.stack(output_inner, dim=1)
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            layer_hidden_list.append((h, c))

        return layer_output_list[-1], layer_hidden_list

    def _init_hidden(self, batch_size, height, width):
        init_states = []
        for i in range(self.num_layers):
            h = torch.zeros(batch_size, self.hidden_dim, height, width).to(device)
            c = torch.zeros(batch_size, self.hidden_dim, height, width).to(device)
            init_states.append((h, c))
        return init_states


In [ ]:

class AttentionMask(nn.Module):
    """CNN + Soft Attention for headlight suppression"""

    def __init__(self, in_channels):
        super(AttentionMask, self).__init__()

        self.attention_net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 4, 3, 1, 1),
            nn.BatchNorm2d(in_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 4, in_channels // 8, 3, 1, 1),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, 1, 1, 1, 0),
            nn.Sigmoid()
        )

    def forward(self, x):
        """Generate attention mask and apply to features"""
        attention_map = self.attention_net(x)
        attended_features = x * attention_map
        return attended_features, attention_map

In [ ]:
class MultiScaleFusion(nn.Module):
    """Multi-scale feature fusion with residual connections"""

    def __init__(self, in_channels):
        super(MultiScaleFusion, self).__init__()

        # Multi-scale convolutions
        self.conv1x1 = nn.Conv2d(in_channels, in_channels // 4, 1, 1, 0)
        self.conv3x3 = nn.Conv2d(in_channels, in_channels // 4, 3, 1, 1)
        self.conv5x5 = nn.Conv2d(in_channels, in_channels // 4, 5, 1, 2)
        self.conv7x7 = nn.Conv2d(in_channels, in_channels // 4, 7, 1, 3)

        # Fusion layer
        self.fusion = nn.Conv2d(in_channels, in_channels, 1, 1, 0)
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        """Fuse multi-scale features"""
        # Multi-scale processing
        x1 = self.conv1x1(x)
        x2 = self.conv3x3(x)
        x3 = self.conv5x5(x)
        x4 = self.conv7x7(x)

        # Concatenate and fuse
        multi_scale = torch.cat([x1, x2, x3, x4], dim=1)
        fused = self.fusion(multi_scale)
        fused = self.bn(fused)

        # Residual connection
        output = self.relu(fused + x)
        return output


In [ ]:
class DetectionHead(nn.Module):
    """YOLOv8-style detection head"""

    def __init__(self, in_channels, num_classes=1, num_anchors=3):
        super(DetectionHead, self).__init__()

        self.num_classes = num_classes
        self.num_anchors = num_anchors

        # Classification head
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, 1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, num_anchors * num_classes, 1, 1, 0)
        )

        # Regression head
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, 1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, num_anchors * 4, 1, 1, 0)  # 4 for bbox coords
        )

        # Objectness head
        self.obj_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, 1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, num_anchors, 1, 1, 0)
        )

    def forward(self, x):
        cls_pred = self.cls_head(x)
        reg_pred = self.reg_head(x)
        obj_pred = self.obj_head(x)

        return cls_pred, reg_pred, obj_pred


In [ ]:
class HybridSpatioTemporalYOLO(nn.Module):
    """Complete Hybrid SpatioTemporal YOLO model"""

    def __init__(self, num_classes=1, sequence_length=8):
        super(HybridSpatioTemporalYOLO, self).__init__()

        self.sequence_length = sequence_length
        self.num_classes = num_classes

        # Module 1: Image Enhancement (handled in preprocessing)
        self.enhancer = ImageEnhancer()

        # Module 2: YOLOv8 Backbone
        self.backbone = YOLOv8Backbone()

        # Module 3: Temporal Modeling (ConvLSTM for each FPN level)
        self.temporal_models = nn.ModuleList([
            ConvLSTM(256, 256, (3, 3)),
            ConvLSTM(512, 512, (3, 3)),
            ConvLSTM(1024, 1024, (3, 3))
        ])

        # Module 4: Attention Masking
        self.attention_masks = nn.ModuleList([
            AttentionMask(256),
            AttentionMask(512),
            AttentionMask(1024)
        ])

        # Module 5: Multi-Scale Feature Fusion
        self.feature_fusion = nn.ModuleList([
            MultiScaleFusion(256),
            MultiScaleFusion(512),
            MultiScaleFusion(1024)
        ])

        # Module 6: Detection Heads
        self.detection_heads = nn.ModuleList([
            DetectionHead(256, num_classes),
            DetectionHead(512, num_classes),
            DetectionHead(1024, num_classes)
        ])

        # Feature pyramid neck
        self.neck = self._build_neck()

    def _build_neck(self):
        """Build FPN neck"""
        return nn.ModuleDict({
            'lateral_convs': nn.ModuleList([
                nn.Conv2d(256, 256, 1, 1, 0),
                nn.Conv2d(512, 256, 1, 1, 0),
                nn.Conv2d(1024, 256, 1, 1, 0)
            ]),
            'fpn_convs': nn.ModuleList([
                nn.Conv2d(256, 256, 3, 1, 1),
                nn.Conv2d(256, 256, 3, 1, 1),
                nn.Conv2d(256, 256, 3, 1, 1)
            ])
        })

    def forward(self, x):
        """
        Forward pass
        x: (batch, seq_len, channels, height, width)
        """
        batch_size, seq_len, c, h, w = x.shape

        # Process each frame through backbone
        backbone_features = []
        for t in range(seq_len):
            frame_features = self.backbone(x[:, t])
            backbone_features.append(frame_features)

        # Reorganize for temporal modeling: [level][time][batch, c, h, w]
        temporal_input = []
        for level in range(3):  # 3 FPN levels
            level_sequence = torch.stack([backbone_features[t][level] for t in range(seq_len)], dim=1)
            temporal_input.append(level_sequence)

        # Apply temporal modeling to each FPN level
        temporal_features = []
        for level in range(3):
            temp_out, _ = self.temporal_models[level](temporal_input[level])
            # Take last timestep for detection
            temporal_features.append(temp_out[:, -1])

        # Apply attention masking
        attended_features = []
        attention_maps = []
        for level in range(3):
            attended, att_map = self.attention_masks[level](temporal_features[level])
            attended_features.append(attended)
            attention_maps.append(att_map)

        # Apply feature fusion
        fused_features = []
        for level in range(3):
            fused = self.feature_fusion[level](attended_features[level])
            fused_features.append(fused)

        # Detection heads
        predictions = []
        for level in range(3):
            cls_pred, reg_pred, obj_pred = self.detection_heads[level](fused_features[level])
            predictions.append((cls_pred, reg_pred, obj_pred))

        return predictions, attention_maps


In [ ]:
class NighttimeVehicleDataset(Dataset):
    """Dataset for nighttime vehicle detection"""

    def __init__(self, data_path, sequence_length=8, transform=None, is_train=True):
        self.data_path = data_path
        self.sequence_length = sequence_length
        self.transform = transform
        self.is_train = is_train
        self.enhancer = ImageEnhancer()

        # Load data paths (modify based on your data structure)
        self.sequences = self._load_sequences()

    def _load_sequences(self):
        """Load video sequences and annotations"""
        # Placeholder - implement based on your data structure
        sequences = []

        # Example structure:
        # sequences = [
        #     {
        #         'frames': ['path1.jpg', 'path2.jpg', ...],
        #         'annotations': [bbox_data1, bbox_data2, ...]
        #     }
        # ]

        # For demo, create dummy data
        for i in range(100):  # 100 dummy sequences
            seq_data = {
                'frames': [f'dummy_frame_{i}_{j}.jpg' for j in range(self.sequence_length)],
                'annotations': [self._dummy_annotation() for _ in range(self.sequence_length)]
            }
            sequences.append(seq_data)

        return sequences

    def _dummy_annotation(self):
        """Create dummy annotation for testing"""
        return {
            'boxes': torch.rand(2, 4) * 100,  # 2 random boxes
            'labels': torch.ones(2, dtype=torch.long),
            'area': torch.rand(2) * 1000,
            'iscrowd': torch.zeros(2, dtype=torch.long)
        }

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]

        # Load and process frames
        frames = []
        for frame_path in sequence['frames']:
            # For demo, create dummy frame
            frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

            # Apply image enhancement
            enhanced_frame = self.enhancer.enhance_frame(frame)

            # Convert to tensor
            if self.transform:
                enhanced_frame = self.transform(enhanced_frame)
            else:
                enhanced_frame = torch.from_numpy(enhanced_frame).permute(2, 0, 1).float() / 255.0

            frames.append(enhanced_frame)

        frames = torch.stack(frames)

        # Process annotations
        annotations = sequence['annotations'][-1]  # Use last frame annotation

        return frames, annotations

In [ ]:
class HybridLoss(nn.Module):
    """Multi-component hybrid loss function"""

    def __init__(self, num_classes=1):
        super(HybridLoss, self).__init__()
        self.num_classes = num_classes
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.mse_loss = nn.MSELoss()

    def forward(self, predictions, targets, attention_maps=None):
        """
        Calculate hybrid loss:
        1. Detection loss (classification + regression + objectness)
        2. Temporal consistency loss
        3. Attention mask loss
        """
        total_loss = 0
        loss_dict = {}

        # Detection loss for each FPN level
        detection_loss = 0
        for level, (cls_pred, reg_pred, obj_pred) in enumerate(predictions):
            # Simplified loss calculation (implement proper YOLO loss)
            # Classification loss
            cls_loss = self.bce_loss(cls_pred, torch.zeros_like(cls_pred))

            # Regression loss
            reg_loss = self.mse_loss(reg_pred, torch.zeros_like(reg_pred))

            # Objectness loss
            obj_loss = self.bce_loss(obj_pred, torch.zeros_like(obj_pred))

            detection_loss += cls_loss + reg_loss + obj_loss

        total_loss += detection_loss
        loss_dict['detection_loss'] = detection_loss.item()

        # Temporal consistency loss (simplified)
        temporal_loss = 0
        if len(predictions) > 1:
            for i in range(len(predictions) - 1):
                curr_pred = predictions[i][0]  # Current frame prediction
                next_pred = predictions[i + 1][0]  # Next frame prediction
                temporal_loss += self.mse_loss(curr_pred, next_pred) * 0.1

        total_loss += temporal_loss
        loss_dict['temporal_loss'] = temporal_loss.item() if temporal_loss != 0 else 0

        # Attention mask loss (encourage sparsity)
        if attention_maps is not None:
            attention_loss = 0
            for att_map in attention_maps:
                # L1 regularization for sparsity
                attention_loss += torch.mean(torch.abs(att_map)) * 0.01

            total_loss += attention_loss
            loss_dict['attention_loss'] = attention_loss.item()

        loss_dict['total_loss'] = total_loss.item()
        return total_loss, loss_dict


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, epoch):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    running_losses = {'detection_loss': 0, 'temporal_loss': 0, 'attention_loss': 0}

    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')

    for batch_idx, (sequences, targets) in enumerate(pbar):
        sequences = sequences.to(device)

        optimizer.zero_grad()

        # Forward pass
        predictions, attention_maps = model(sequences)

        # Calculate loss
        loss, loss_dict = criterion(predictions, targets, attention_maps)

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

        optimizer.step()

        # Update running losses
        running_loss += loss.item()
        for key in running_losses:
            running_losses[key] += loss_dict.get(key, 0)

        # Update progress bar
        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Det': f'{loss_dict.get("detection_loss", 0):.4f}',
            'Temp': f'{loss_dict.get("temporal_loss", 0):.4f}',
            'Att': f'{loss_dict.get("attention_loss", 0):.4f}'
        })

    # Calculate average losses
    avg_loss = running_loss / len(dataloader)
    avg_losses = {key: val / len(dataloader) for key, val in running_losses.items()}

    return avg_loss, avg_losses

def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    running_losses = {'detection_loss': 0, 'temporal_loss': 0, 'attention_loss': 0}

    with torch.no_grad():
        for sequences, targets in tqdm(dataloader, desc='Validation'):
            sequences = sequences.to(device)

            predictions, attention_maps = model(sequences)
            loss, loss_dict = criterion(predictions, targets, attention_maps)

            running_loss += loss.item()
            for key in running_losses:
                running_losses[key] += loss_dict.get(key, 0)

    avg_loss = running_loss / len(dataloader)
    avg_losses = {key: val / len(dataloader) for key, val in running_losses.items()}

    return avg_loss, avg_losses


In [ ]:
# Improved HybridSpatioTemporalYOLO Training with Enhanced Performance

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import os
import random
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time
import warnings
import matplotlib.pyplot as plt
from collections import defaultdict
warnings.filterwarnings('ignore')

class ImprovedHybridLoss(nn.Module):
    """
    Enhanced hybrid loss function with better convergence properties
    """
    def __init__(self, num_classes=1, detection_weight=1.0, temporal_weight=0.5, attention_weight=0.1):
        super(ImprovedHybridLoss, self).__init__()
        self.num_classes = num_classes
        self.detection_weight = detection_weight
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight

        # More appropriate loss functions
        self.focal_loss = FocalLoss(alpha=0.25, gamma=2.0)
        self.smooth_l1 = nn.SmoothL1Loss()
        self.consistency_loss = nn.MSELoss()

    def forward(self, predictions, targets, attention_maps):
        """
        Enhanced loss calculation with meaningful target generation
        """
        device = predictions[0].device if len(predictions) > 0 else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 1. Enhanced detection loss with realistic targets
        detection_loss = 0.0
        for i, pred in enumerate(predictions):
            if isinstance(pred, tuple):
                pred = pred[0]

            if len(pred.shape) == 5:
                pred = pred[:, -1]  # Use last timestep

            # Create more realistic targets based on pred shape
            batch_size = pred.shape[0]
            # Simulate object detection targets with some positive examples
            targets_realistic = torch.zeros_like(pred)

            # Add some positive targets (simulate objects)
            for b in range(batch_size):
                num_objects = random.randint(1, 3)  # 1-3 objects per image
                for _ in range(num_objects):
                    if len(pred.shape) == 4:  # [batch, channels, h, w]
                        h, w = pred.shape[2], pred.shape[3]
                        obj_h = random.randint(0, h-1)
                        obj_w = random.randint(0, w-1)
                        targets_realistic[b, :, obj_h, obj_w] = torch.rand(pred.shape[1]) * 0.8 + 0.2
                    elif len(pred.shape) == 3:  # [batch, h, w]
                        h, w = pred.shape[1], pred.shape[2]
                        obj_h = random.randint(0, h-1)
                        obj_w = random.randint(0, w-1)
                        targets_realistic[b, obj_h, obj_w] = random.uniform(0.2, 0.9)

            # Use focal loss for better handling of class imbalance
            detection_loss += self.focal_loss(pred, targets_realistic)

        # 2. Temporal consistency loss
        temporal_loss = 0.0
        if len(predictions) > 1:
            for i in range(len(predictions) - 1):
                pred_curr = predictions[i]
                pred_next = predictions[i + 1]

                if isinstance(pred_curr, tuple):
                    pred_curr = pred_curr[0]
                if isinstance(pred_next, tuple):
                    pred_next = pred_next[0]

                # Ensure same shape for consistency
                if pred_curr.shape == pred_next.shape:
                    temporal_loss += self.consistency_loss(pred_curr, pred_next)

        # 3. Attention regularization with sparsity
        attention_loss = 0.0
        for attention_map in attention_maps:
            if isinstance(attention_map, tuple):
                attention_map = attention_map[0]

            # Encourage sparsity in attention
            attention_loss += torch.mean(torch.abs(attention_map))  # L1 for sparsity
            # Encourage focused attention
            attention_loss += -torch.mean(attention_map * torch.log(attention_map + 1e-8))  # Entropy

        # Combine losses with adaptive weights
        total_loss = (self.detection_weight * detection_loss +
                     self.temporal_weight * temporal_loss +
                     self.attention_weight * attention_loss)

        loss_dict = {
            'detection_loss': detection_loss.item() if isinstance(detection_loss, torch.Tensor) else 0.0,
            'temporal_loss': temporal_loss.item() if isinstance(temporal_loss, torch.Tensor) else 0.0,
            'attention_loss': attention_loss.item() if isinstance(attention_loss, torch.Tensor) else 0.0,
            'total_loss': total_loss.item()
        }

        return total_loss, loss_dict

class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class EnhancedModel(nn.Module):
    """Enhanced model with better architecture"""
    def __init__(self, num_classes, sequence_length):
        super().__init__()
        self.sequence_length = sequence_length

        # Improved feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((13, 13))
        )

        # Temporal processing
        self.temporal_conv = nn.Conv3d(128, 256, (3, 3, 3), padding=(1, 1, 1))
        self.temporal_norm = nn.BatchNorm3d(256)

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Conv2d(256, 128, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, 1),
            nn.Sigmoid()
        )

        # Detection heads
        self.detection_head = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, 1)
        )

        # Classification head for evaluation
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: [batch, sequence, channels, height, width]
        batch_size, seq_len = x.shape[:2]

        # Process each frame
        features = []
        for t in range(seq_len):
            frame_features = self.feature_extractor(x[:, t])
            features.append(frame_features)

        # Stack temporal features
        temporal_features = torch.stack(features, dim=2)  # [batch, channels, time, h, w]

        # Temporal processing
        temporal_out = self.temporal_conv(temporal_features)
        temporal_out = self.temporal_norm(temporal_out)
        temporal_out = torch.relu(temporal_out)

        # Use last timestep for detection
        final_features = temporal_out[:, :, -1]  # [batch, channels, h, w]

        # Generate attention maps
        attention_maps = self.attention(final_features)

        # Apply attention
        attended_features = final_features * attention_maps

        # Detection output
        detection_output = self.detection_head(attended_features)

        # Classification output for evaluation
        classification_output = self.classifier(attended_features)

        predictions = [detection_output, classification_output]
        attention_maps_list = [attention_maps]

        return predictions, attention_maps_list

def calculate_enhanced_metrics(predictions, targets, threshold=0.5):
    """Calculate comprehensive performance metrics"""
    if torch.is_tensor(predictions):
        predictions = predictions.cpu().numpy()
    if torch.is_tensor(targets):
        targets = targets.cpu().numpy()

    # Apply sigmoid if needed
    if predictions.max() > 1.0 or predictions.min() < 0.0:
        predictions = 1 / (1 + np.exp(-predictions))  # Sigmoid

    # Binary predictions
    binary_preds = (predictions > threshold).astype(int)

    # Flatten arrays
    targets_flat = targets.flatten()
    preds_flat = binary_preds.flatten()

    # Calculate metrics
    accuracy = accuracy_score(targets_flat, preds_flat)
    f1 = f1_score(targets_flat, preds_flat, average='weighted', zero_division=0)
    precision = precision_score(targets_flat, preds_flat, average='weighted', zero_division=0)
    recall = recall_score(targets_flat, preds_flat, average='weighted', zero_division=0)

    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

class EnhancedDataset:
    """Enhanced dataset with more realistic data patterns"""
    def __init__(self, size, sequence_length, transform=None, mode='train'):
        self.size = size
        self.sequence_length = sequence_length
        self.transform = transform
        self.mode = mode

        # Generate more structured data patterns
        self.patterns = self._generate_patterns()

    def _generate_patterns(self):
        """Generate structured patterns for better learning"""
        patterns = []
        for i in range(self.size):
            # Create different types of patterns
            pattern_type = i % 4
            if pattern_type == 0:
                # Moving object pattern
                pattern = self._create_moving_object_pattern()
            elif pattern_type == 1:
                # Static object pattern
                pattern = self._create_static_object_pattern()
            elif pattern_type == 2:
                # Multiple objects pattern
                pattern = self._create_multiple_objects_pattern()
            else:
                # Background pattern
                pattern = self._create_background_pattern()

            patterns.append(pattern)
        return patterns

    def _create_moving_object_pattern(self):
        """Create a pattern with moving objects"""
        sequence = torch.randn(self.sequence_length, 3, 416, 416) * 0.1
        target = torch.zeros(1)

        # Add moving bright spot
        for t in range(self.sequence_length):
            x_pos = int(50 + t * 30) % 350
            y_pos = int(50 + t * 20) % 350
            sequence[t, :, y_pos:y_pos+20, x_pos:x_pos+20] += 0.8

        target[0] = 1.0  # Has object
        return sequence, target

    def _create_static_object_pattern(self):
        """Create a pattern with static objects"""
        sequence = torch.randn(self.sequence_length, 3, 416, 416) * 0.1
        target = torch.zeros(1)

        # Add static bright regions
        for t in range(self.sequence_length):
            sequence[t, :, 100:150, 100:150] += 0.6
            sequence[t, :, 200:250, 200:250] += 0.4

        target[0] = 1.0  # Has object
        return sequence, target

    def _create_multiple_objects_pattern(self):
        """Create a pattern with multiple objects"""
        sequence = torch.randn(self.sequence_length, 3, 416, 416) * 0.1
        target = torch.zeros(1)

        # Add multiple objects
        for t in range(self.sequence_length):
            # Object 1
            x1 = int(80 + t * 10) % 300
            sequence[t, :, 50:80, x1:x1+30] += 0.7

            # Object 2
            y2 = int(150 + t * 15) % 300
            sequence[t, :, y2:y2+25, 250:280] += 0.5

        target[0] = 1.0  # Has objects
        return sequence, target

    def _create_background_pattern(self):
        """Create a background pattern without objects"""
        sequence = torch.randn(self.sequence_length, 3, 416, 416) * 0.2
        target = torch.zeros(1)

        # Add some texture but no distinct objects
        for t in range(self.sequence_length):
            sequence[t] += torch.sin(torch.arange(416).float().view(1, 1, -1) * 0.1) * 0.1
            sequence[t] += torch.cos(torch.arange(416).float().view(1, -1, 1) * 0.1) * 0.1

        target[0] = 0.0  # No object
        return sequence, target

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.patterns[idx]

def train_epoch_enhanced(model, dataloader, optimizer, criterion, device, epoch):
    """Enhanced training epoch with better metrics"""
    model.train()

    running_loss = 0.0
    running_losses = defaultdict(float)
    all_predictions = []
    all_targets = []

    for batch_idx, (sequences, targets) in enumerate(dataloader):
        sequences = sequences.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        try:
            predictions, attention_maps = model(sequences)
            loss, loss_dict = criterion(predictions, targets, attention_maps)

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Update metrics
            running_loss += loss.item()
            for key, value in loss_dict.items():
                running_losses[key] += value

            # Collect predictions for evaluation (use classification output)
            if len(predictions) > 1:
                class_pred = predictions[1]  # Classification output
                all_predictions.extend(class_pred.cpu().detach().numpy().flatten())
                all_targets.extend(targets.cpu().numpy().flatten())

            # Print progress
            if batch_idx % 10 == 0:
                print(f'Epoch {epoch}, Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}')

        except Exception as e:
            print(f"Error in batch {batch_idx}: {str(e)}")
            continue

    # Calculate epoch metrics
    avg_loss = running_loss / len(dataloader)
    avg_losses = {key: value / len(dataloader) for key, value in running_losses.items()}

    # Calculate performance metrics
    if len(all_predictions) > 0:
        metrics = calculate_enhanced_metrics(np.array(all_predictions), np.array(all_targets))
    else:
        metrics = {'accuracy': 0.0, 'f1_score': 0.0, 'precision': 0.0, 'recall': 0.0}

    return avg_loss, avg_losses, metrics

def validate_epoch_enhanced(model, dataloader, criterion, device):
    """Enhanced validation epoch"""
    model.eval()

    running_loss = 0.0
    running_losses = defaultdict(float)
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for sequences, targets in dataloader:
            sequences = sequences.to(device)
            targets = targets.to(device)

            try:
                predictions, attention_maps = model(sequences)
                loss, loss_dict = criterion(predictions, targets, attention_maps)

                # Update metrics
                running_loss += loss.item()
                for key, value in loss_dict.items():
                    running_losses[key] += value

                # Collect predictions for evaluation
                if len(predictions) > 1:
                    class_pred = predictions[1]  # Classification output
                    all_predictions.extend(class_pred.cpu().numpy().flatten())
                    all_targets.extend(targets.cpu().numpy().flatten())

            except Exception as e:
                print(f"Error in validation batch: {str(e)}")
                continue

    # Calculate epoch metrics
    avg_loss = running_loss / len(dataloader)
    avg_losses = {key: value / len(dataloader) for key, value in running_losses.items()}

    # Calculate performance metrics
    if len(all_predictions) > 0:
        metrics = calculate_enhanced_metrics(np.array(all_predictions), np.array(all_targets))
    else:
        metrics = {'accuracy': 0.0, 'f1_score': 0.0, 'precision': 0.0, 'recall': 0.0}

    return avg_loss, avg_losses, metrics

def main_enhanced():
    """Enhanced main training function"""

    # Improved hyperparameters
    BATCH_SIZE = 8
    SEQUENCE_LENGTH = 8
    NUM_EPOCHS = 100  # Increased epochs for better convergence
    LEARNING_RATE = 0.001
    NUM_CLASSES = 1

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("🚀 Starting Enhanced HybridSpatioTemporalYOLO Training")
    print(f"Device: {device}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Sequence Length: {SEQUENCE_LENGTH}")
    print(f"Number of Epochs: {NUM_EPOCHS}")

    # Create save directory
    save_dir = "enhanced_hybrid_yolo_results"
    os.makedirs(save_dir, exist_ok=True)

    # Enhanced datasets
    train_dataset = EnhancedDataset(size=500, sequence_length=SEQUENCE_LENGTH, mode='train')
    val_dataset = EnhancedDataset(size=100, sequence_length=SEQUENCE_LENGTH, mode='val')

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")

    # Initialize enhanced model
    model = EnhancedModel(num_classes=NUM_CLASSES, sequence_length=SEQUENCE_LENGTH).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Enhanced loss function and optimizer
    criterion = ImprovedHybridLoss(
        num_classes=NUM_CLASSES,
        detection_weight=1.0,
        temporal_weight=0.3,
        attention_weight=0.1
    )

    # Better optimizer with proper weight decay
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4,
        betas=(0.9, 0.999)
    )

    # Enhanced scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6
    )

    # Tracking variables
    best_val_loss = float('inf')
    best_f1_score = 0.0
    best_accuracy = 0.0

    # Training history
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    print("\n" + "="*80)
    print("🏋️ STARTING ENHANCED TRAINING")
    print("="*80)

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        print("-" * 50)

        epoch_start_time = time.time()

        try:
            # Train
            train_loss, train_loss_dict, train_metrics = train_epoch_enhanced(
                model, train_loader, optimizer, criterion, device, epoch
            )

            # Validate
            val_loss, val_loss_dict, val_metrics = validate_epoch_enhanced(
                model, val_loader, criterion, device
            )

            # Learning rate step
            scheduler.step()
            current_lr = optimizer.param_groups[0]['lr']

            epoch_time = time.time() - epoch_start_time

            # Store history
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accuracies.append(train_metrics['accuracy'])
            val_accuracies.append(val_metrics['accuracy'])

            # Print detailed results
            print(f"✅ Epoch {epoch} completed in {epoch_time:.2f}s")
            print(f"Loss - Train: {train_loss:.4f} | Val: {val_loss:.4f}")
            print(f"Detailed Losses:")
            print(f"  Detection: {train_loss_dict.get('detection_loss', 0):.4f} | {val_loss_dict.get('detection_loss', 0):.4f}")
            print(f"  Temporal:  {train_loss_dict.get('temporal_loss', 0):.4f} | {val_loss_dict.get('temporal_loss', 0):.4f}")
            print(f"  Attention: {train_loss_dict.get('attention_loss', 0):.4f} | {val_loss_dict.get('attention_loss', 0):.4f}")
            print(f"Metrics - Train | Val")
            print(f"  Accuracy:  {train_metrics['accuracy']:.4f} | {val_metrics['accuracy']:.4f}")
            print(f"  F1 Score:  {train_metrics['f1_score']:.4f} | {val_metrics['f1_score']:.4f}")
            print(f"  Precision: {train_metrics['precision']:.4f} | {val_metrics['precision']:.4f}")
            print(f"  Recall:    {train_metrics['recall']:.4f} | {val_metrics['recall']:.4f}")
            print(f"Learning Rate: {current_lr:.6f}")

            # Save best model
            if val_metrics['f1_score'] > best_f1_score:
                best_f1_score = val_metrics['f1_score']
                best_val_loss = val_loss
                best_accuracy = val_metrics['accuracy']

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_val_loss': best_val_loss,
                    'best_f1_score': best_f1_score,
                    'best_accuracy': best_accuracy,
                    'best_precision': val_metrics['precision'],
                    'best_recall': val_metrics['recall'],
                    'train_history': {
                        'train_losses': train_losses,
                        'val_losses': val_losses,
                        'train_accuracies': train_accuracies,
                        'val_accuracies': val_accuracies
                    }
                }, os.path.join(save_dir, 'best_enhanced_hybrid_yolo.pth'))

                print(f"🏆 NEW BEST MODEL! F1: {val_metrics['f1_score']:.4f}, Acc: {val_metrics['accuracy']:.4f}")

            # Early stopping check
            if epoch > 20 and val_metrics['accuracy'] > 0.95:
                print(f"🎯 Early stopping - High accuracy achieved: {val_metrics['accuracy']:.4f}")
                break

        except Exception as e:
            print(f"⚠️ Error in epoch {epoch}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue

    print("\n" + "="*80)
    print("🎉 ENHANCED TRAINING COMPLETED!")
    print("="*80)
    print(f"🏆 BEST RESULTS:")
    print(f"   Best F1 Score: {best_f1_score:.4f}")
    print(f"   Best Accuracy: {best_accuracy:.4f}")
    print(f"   Best Val Loss: {best_val_loss:.4f}")

    # Plot training history
    try:
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 3, 2)
        plt.plot(train_accuracies, label='Train Accuracy')
        plt.plot(val_accuracies, label='Val Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 3, 3)
        epochs = range(1, len(train_losses) + 1)
        improvement = np.array(val_accuracies) - np.array(train_accuracies)
        plt.plot(epochs, improvement, label='Val - Train Accuracy')
        plt.title('Generalization Gap')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy Difference')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'training_history.png'), dpi=300, bbox_inches='tight')
        plt.show()

    except Exception as e:
        print(f"Could not generate plots: {e}")

    return model, best_f1_score, best_accuracy, best_val_loss

# Test the enhanced solution
if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    print("🌟 Enhanced HybridSpatioTemporalYOLO Training")
    print("="*80)

    try:
        # Run enhanced training
        print("\n🚀 Starting enhanced training...")
        trained_model, best_f1, best_accuracy, best_loss = main_enhanced()

        print(f"\n🎊 FINAL ENHANCED RESULTS:")
        print(f"   🥇 Best F1 Score: {best_f1:.4f}")
        print(f"   🎯 Best Accuracy: {best_accuracy:.4f}")
        print(f"   📉 Best Loss: {best_loss:.4f}")
        print(f"\n📁 Enhanced models and plots saved in 'enhanced_hybrid_yolo_results'")

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

🌟 Enhanced HybridSpatioTemporalYOLO Training

🚀 Starting enhanced training...
🚀 Starting Enhanced HybridSpatioTemporalYOLO Training
Device: cuda
Batch Size: 8
Sequence Length: 8
Number of Epochs: 100
